In [ ]:
TEAM_NAME = 'Group_01'


In [ ]:
# The matcher uses the three inventories provided by the grader.


In [ ]:
# The matcher uses only Python standard-library modules.


In [ ]:
from collections.abc import Iterator
from dataclasses import dataclass
import re
import unicodedata
import math
from typing import Any
from collections.abc import Sequence, Set
from bisect import bisect_left
from dataclasses import replace


In [ ]:
# Generated by tools.build_submission from the maintained matcher.
class Solution:

    def __init__(
        self,
        list_province: list[str],
        list_ward: list[str],
        list_street: list[str],
    ):
        self._matcher = self.AddressMatcher(
            streets=list_street, wards=list_ward, provinces=list_province,
        )

    def process(self, s: str):
        return self._matcher.identify(s)


    # normalize.py
    @dataclass(frozen=True, slots=True)
    class Token:
        """A token with raw text, normalized forms, and character offsets in the source string."""
        text: str
        nfc: str
        folded: str
        start: int
        end: int

    @staticmethod
    def is_combining_mark(c: str) -> bool:
        """Check if character is a Unicode combining diacritical mark (category Mn, Mc, Me)."""
        return unicodedata.category(c).startswith('M')

    @staticmethod
    def fold_accents(text: str) -> str:
        """Casefold and fold all Vietnamese and Latin diacritics, including đ/Đ -> d."""
        cf = unicodedata.normalize('NFC', text).casefold()
        if cf.isascii():
            return cf
        cf = cf.replace('đ', 'd')
        nfd = unicodedata.normalize('NFD', cf)
        if len(nfd) >= 256:
            marks = dict.fromkeys((ord(c) for c in set(nfd) if unicodedata.category(c) == 'Mn'))
            stripped = nfd.translate(marks) if marks else nfd
        else:
            stripped = ''.join((c for c in nfd if unicodedata.category(c) != 'Mn'))
        return unicodedata.normalize('NFC', stripped)

    @staticmethod
    def normalize_nfc(text: str) -> str:
        """Normalize string to Unicode NFC form."""
        return unicodedata.normalize('NFC', text)

    BASIC_TOKEN_PATTERN = re.compile('[^\\W_]+(?:/[^\\W_]+)*')

    @staticmethod
    def _token_spans(text: str) -> Iterator[tuple[int, int]]:
        """Scan shared address/label boundaries without allocating normalized tokens."""
        n = len(text)
        if n >= 256 and (text.isascii() or not any((Solution.is_combining_mark(c) for c in set(text)))):
            for match in Solution.BASIC_TOKEN_PATTERN.finditer(text):
                yield match.span()
            return
        i = 0
        while i < n:
            if not text[i].isalnum():
                i += 1
                continue
            token_start = i
            while i < n:
                while i < n and (text[i].isalnum() or Solution.is_combining_mark(text[i])):
                    i += 1
                if i < n and text[i] == '/' and (i + 1 < n) and text[i + 1].isalnum():
                    i += 1
                else:
                    break
            yield (token_start, i)

    @staticmethod
    def tokenize(text: str) -> 'list[Token]':
        """Tokenize original text with accurate source offsets, including NFD marks.

        Slash-connected forms remain whole (3/2 cannot match inside 53/2).
        """
        tokens: 'list[Token]' = []
        forms: dict[str, tuple[str, str]] = {}
        for token_start, token_end in Solution._token_spans(text):
            raw = text[token_start:token_end]
            if raw not in forms:
                nfc = unicodedata.normalize('NFC', raw).casefold()
                forms[raw] = (nfc, Solution.fold_accents(nfc))
            nfc, folded = forms[raw]
            tokens.append(Solution.Token(raw, nfc, folded, token_start, token_end))
        return tokens

    @staticmethod
    def tokenize_label(label: str, *, token_cache: dict[str, tuple[str, str]] | None=None) -> tuple[tuple[str, ...], tuple[str, ...]]:
        """Normalize labels without allocating source-offset Token objects.

        A catalog-local cache reuses repeated words while building one dictionary;
        it is discarded after construction and never caches request results.
        """
        nfc_words, folded_words = ([], [])
        for start, end in Solution._token_spans(label):
            raw = label[start:end]
            forms = token_cache.get(raw) if token_cache is not None else None
            if forms is None:
                nfc = unicodedata.normalize('NFC', raw).casefold()
                forms = (nfc, Solution.fold_accents(nfc))
                if token_cache is not None:
                    token_cache[raw] = forms
            nfc_words.append(forms[0])
            folded_words.append(forms[1])
        return (tuple(nfc_words), tuple(folded_words))

    # model.py
    @dataclass(frozen=True, slots=True)
    class MatcherConfig:
        """Configuration parameters for address matching and candidate selection.

        Attributes:
            top_k: Maximum candidate pool size per field for joint selection (default 8).
            ambiguity_margin: Score delta within which alternative labels trigger abstention (default 0.0).
            min_score: Minimum evidence score required for candidate acceptance (default 10.0).
            enable_fuzzy: Flag for approximate matching stages (default True in Stage D).
            enable_compact: Enable dictionary-constrained whitespace recovery (default True).
            enable_typing: Enable candidate-specific Telex/VNI recovery (default True).
            max_fuzzy_spans: Maximum component spans evaluated for approximate search (default 128).
            max_candidates_per_span: Maximum shortlisted candidates per span (default 64).
            max_verifications: Maximum thresholded OSA verification calls (default 2048).
            max_postings: Maximum trigram postings entries visited (default 12000).
            max_edits: Maximum allowable edit distance (default 2).
        """
        top_k: int = 8
        ambiguity_margin: float = 0.0
        min_score: float = 10.0
        enable_fuzzy: bool = True
        enable_compact: bool = True
        enable_typing: bool = True
        max_fuzzy_spans: int = 128
        max_candidates_per_span: int = 64
        max_verifications: int = 2048
        max_postings: int = 12000
        max_edits: int = 2

        def __post_init__(self) -> None:
            if isinstance(self.top_k, bool) or not isinstance(self.top_k, int):
                raise TypeError(f'top_k must be an int, got {type(self.top_k).__name__}')
            if self.top_k <= 0:
                raise ValueError(f'top_k must be a positive integer, got {self.top_k}')
            if isinstance(self.ambiguity_margin, bool) or not isinstance(self.ambiguity_margin, (int, float)):
                raise TypeError(f'ambiguity_margin must be a float or int, got {type(self.ambiguity_margin).__name__}')
            if not math.isfinite(self.ambiguity_margin):
                raise ValueError(f'ambiguity_margin must be finite, got {self.ambiguity_margin}')
            if self.ambiguity_margin < 0.0:
                raise ValueError(f'ambiguity_margin must be non-negative, got {self.ambiguity_margin}')
            if isinstance(self.min_score, bool) or not isinstance(self.min_score, (int, float)):
                raise TypeError(f'min_score must be a float or int, got {type(self.min_score).__name__}')
            if not math.isfinite(self.min_score):
                raise ValueError(f'min_score must be finite, got {self.min_score}')
            for flag_name in ('enable_fuzzy', 'enable_compact', 'enable_typing'):
                flag_val = getattr(self, flag_name)
                if not isinstance(flag_val, bool):
                    raise TypeError(f'{flag_name} must be a bool, got {type(flag_val).__name__}')
            for bound_name in ('max_fuzzy_spans', 'max_candidates_per_span', 'max_verifications', 'max_postings', 'max_edits'):
                bound_val = getattr(self, bound_name)
                if isinstance(bound_val, bool) or not isinstance(bound_val, int):
                    raise TypeError(f'{bound_name} must be an int, got {type(bound_val).__name__}')
                if bound_val <= 0:
                    raise ValueError(f'{bound_name} must be a positive integer, got {bound_val}')

    @dataclass(frozen=True, slots=True)
    class Candidate:
        """An evidence candidate representing a dictionary entry matched against a source span.

        Attributes:
            entry: Associated immutable CatalogEntry.
            start: Start character offset in the original raw address string.
            end: End character offset (half-open) in the original raw address string.
            path: Retrieval match path ('exact', 'folded', 'fuzzy', etc.).
            cost: Verification edit distance or penalty cost (default 0).
            score: Additive evidence score computed from match quality and context.
            context_marker: Whether an explicit preceding field marker supported this match.
            reasons: Diagnostic tags and features explaining candidate scoring.
            body_start: Optional source offset of the body after a joined marker.
        """
        entry: 'CatalogEntry'
        start: int
        end: int
        path: str
        cost: int = 0
        score: float = 0.0
        context_marker: bool = False
        reasons: tuple[str, ...] = ()
        body_start: int | None = None

        def __post_init__(self) -> None:
            if self.path not in ('exact', 'folded', 'fuzzy', 'compact', 'telex', 'vni'):
                raise ValueError(f'Invalid candidate path: {self.path!r}')
            if self.start < 0 or self.end < self.start:
                raise ValueError(f'Invalid candidate character span: [{self.start}, {self.end}]')

        @property
        def span(self) -> tuple[int, int]:
            """Return the character offset tuple (start, end)."""
            return (self.start, self.end)

        @property
        def field(self) -> str:
            """Return entry field name ('street', 'ward', 'province')."""
            return self.entry.field

        @property
        def label(self) -> str:
            """Return canonical label text."""
            return self.entry.canonical_label

        def to_dict(self) -> dict[str, Any]:
            """Return JSON-serializable dictionary representation for trace diagnostics."""
            return {'field': self.field, 'label': self.label, 'span': [self.start, self.end], 'path': self.path, 'cost': self.cost, 'score': round(float(self.score), 4)}

    # catalog.py
    LEXICOGRAPHICAL_FIELDS: tuple[str, ...] = ('province', 'street', 'ward')

    WARD_PREFIX_PATTERNS: tuple[tuple[str, ...], ...] = (('thị', 'trấn'), ('thi', 'tran'), ('phường',), ('phuong',), ('p',), ('xã',), ('xa',), ('x',), ('tt',))

    PROVINCE_PREFIX_PATTERNS: tuple[tuple[str, ...], ...] = (('thành', 'phố'), ('thanh', 'pho'), ('tp',), ('tỉnh',), ('tinh',), ('t',))

    STREET_PREFIX_PATTERNS: tuple[tuple[str, ...], ...] = (('đường',), ('duong',), ('đ',), ('d',))

    FIELD_PREFIX_PATTERNS: dict[str, tuple[tuple[str, ...], ...]] = {'ward': WARD_PREFIX_PATTERNS, 'province': PROVINCE_PREFIX_PATTERNS, 'street': STREET_PREFIX_PATTERNS}

    @dataclass(frozen=True, slots=True)
    class CatalogEntry:
        """An immutable dictionary entry with stable ID, field, canonical label, and tokens.

        Attributes:
            entry_id: Stable deterministic integer ID.
            field: Field name ('province', 'street', 'ward').
            canonical_label: Original canonical string label from dictionary.
            nfc_tokens: Casefolded Unicode NFC token sequence of the full label.
            folded_tokens: Accent-folded token sequence of the full label.
            prefix_tokens: Tokens corresponding to an administrative prefix, if present.
            body_nfc_tokens: Casefolded Unicode NFC token sequence of the body.
            body_folded_tokens: Accent-folded token sequence of the body.
            has_prefix: True if canonical label begins with a recognized administrative prefix.
        """
        entry_id: int
        field: str
        canonical_label: str
        nfc_tokens: tuple[str, ...]
        folded_tokens: tuple[str, ...]
        prefix_tokens: tuple[str, ...] = ()
        body_nfc_tokens: tuple[str, ...] = ()
        body_folded_tokens: tuple[str, ...] = ()
        has_prefix: bool = False

    @staticmethod
    def extract_prefix_body(field: str, nfc_tokens: tuple[str, ...], folded_tokens: tuple[str, ...]) -> tuple[bool, tuple[str, ...], tuple[str, ...], tuple[str, ...]]:
        """Extract administrative prefix and body tokens for an entry.

        Returns (has_prefix, prefix_tokens, body_nfc_tokens, body_folded_tokens).
        """
        patterns = Solution.FIELD_PREFIX_PATTERNS.get(field, ())
        for pattern in patterns:
            plen = len(pattern)
            if len(nfc_tokens) > plen:
                if nfc_tokens[:plen] == pattern or folded_tokens[:plen] == pattern:
                    return (True, nfc_tokens[:plen], nfc_tokens[plen:], folded_tokens[plen:])
        return (False, (), nfc_tokens, folded_tokens)

    @staticmethod
    def _validated_field_forms(labels: Any, field_name: str, token_cache: dict[str, tuple[str, str]]) -> list[tuple[str, tuple[str, ...], tuple[str, ...]]]:
        """Validate that labels is a finite sequence or set of non-empty strings.

        Rejects scalar strings, non-sequences, non-string items, and blank strings.
        """
        if isinstance(labels, (str, bytes)):
            raise TypeError(f'{field_name} must be a sequence of strings, not a scalar {type(labels).__name__}')
        if not isinstance(labels, (Sequence, Set)):
            raise TypeError(f'{field_name} must be a finite sequence or set, got {type(labels).__name__}')
        validated = []
        for idx, item in enumerate(labels):
            if not isinstance(item, str):
                raise TypeError(f'Item at index {idx} in {field_name} must be str, got {type(item).__name__}')
            if not item.strip():
                raise ValueError(f'Item at index {idx} in {field_name} cannot be blank or whitespace-only')
            nfc_tokens, folded_tokens = Solution.tokenize_label(item, token_cache=token_cache)
            if not nfc_tokens:
                raise ValueError(f'Item at index {idx} in {field_name} contains no recognizable word or numeric tokens: {item!r}')
            validated.append((item, nfc_tokens, folded_tokens))
        return validated

    class Catalog:
        """Immutable catalog storing entries and collision-safe token lookup indexes."""

        def __init__(self, streets: Sequence[str] | Set[str]=(), wards: Sequence[str] | Set[str]=(), provinces: Sequence[str] | Set[str]=()) -> None:
            token_cache: dict[str, tuple[str, str]] = {}
            unique_by_field = {}
            for field, labels in (('street', streets), ('ward', wards), ('province', provinces)):
                unique_by_field[field] = {label: (nfc, folded) for label, nfc, folded in Solution._validated_field_forms(labels, field + 's', token_cache)}
            entries: 'list[CatalogEntry]' = []
            entry_id = 0
            for field in Solution.LEXICOGRAPHICAL_FIELDS:
                for label, (nfc_tokens, folded_tokens) in sorted(unique_by_field[field].items()):
                    has_prefix, prefix_tokens, body_nfc, body_folded = Solution.extract_prefix_body(field, nfc_tokens, folded_tokens)
                    entry = Solution.CatalogEntry(entry_id=entry_id, field=field, canonical_label=label, nfc_tokens=nfc_tokens, folded_tokens=folded_tokens, prefix_tokens=prefix_tokens, body_nfc_tokens=body_nfc, body_folded_tokens=body_folded, has_prefix=has_prefix)
                    entries.append(entry)
                    entry_id += 1
            self._entries: 'tuple[CatalogEntry, ...]' = tuple(entries)
            exact_map: 'dict[tuple[str, ...], list[CatalogEntry]]' = {}
            folded_map: 'dict[tuple[str, ...], list[CatalogEntry]]' = {}
            body_exact_map: 'dict[tuple[str, ...], list[CatalogEntry]]' = {}
            body_folded_map: 'dict[tuple[str, ...], list[CatalogEntry]]' = {}
            province_initials: 'dict[str, list[CatalogEntry]]' = {}
            lengths: set[int] = set()
            for entry in self._entries:
                exact_map.setdefault(entry.nfc_tokens, []).append(entry)
                folded_map.setdefault(entry.folded_tokens, []).append(entry)
                lengths.add(len(entry.nfc_tokens))
                body_exact_map.setdefault(entry.body_nfc_tokens, []).append(entry)
                body_folded_map.setdefault(entry.body_folded_tokens, []).append(entry)
                lengths.add(len(entry.body_nfc_tokens))
                if entry.field == 'province' and len(entry.body_folded_tokens) >= 2 and all((word.isalpha() for word in entry.body_folded_tokens)):
                    initials = ''.join((word[0] for word in entry.body_folded_tokens))
                    province_initials.setdefault(initials, []).append(entry)
                    lengths.add(1)
            self._exact_index: 'dict[tuple[str, ...], tuple[CatalogEntry, ...]]' = {k: tuple(v) for k, v in exact_map.items()}
            self._folded_index: 'dict[tuple[str, ...], tuple[CatalogEntry, ...]]' = {k: tuple(v) for k, v in folded_map.items()}
            self._body_exact_index: 'dict[tuple[str, ...], tuple[CatalogEntry, ...]]' = {k: tuple(v) for k, v in body_exact_map.items()}
            self._body_folded_index: 'dict[tuple[str, ...], tuple[CatalogEntry, ...]]' = {k: tuple(v) for k, v in body_folded_map.items()}
            self._province_initials = {k: tuple(v) for k, v in province_initials.items()}
            window_lengths: dict[str, set[int]] = {}
            for words in folded_map.keys() | body_folded_map.keys():
                window_lengths.setdefault(words[0], set()).add(len(words))
            for initials in province_initials:
                window_lengths.setdefault(initials, set()).add(1)
            self._window_lengths = {k: tuple(sorted(v)) for k, v in window_lengths.items()}
            self._token_lengths: frozenset[int] = frozenset(lengths)
            self._max_tokens: int = max(self._token_lengths) if self._token_lengths else 0

        @property
        def entries(self) -> 'tuple[CatalogEntry, ...]':
            return self._entries

        @property
        def max_tokens(self) -> int:
            return self._max_tokens

        @property
        def token_lengths(self) -> frozenset[int]:
            return self._token_lengths

        def lookup_exact(self, nfc_tokens: tuple[str, ...]) -> 'tuple[CatalogEntry, ...]':
            """Look up all entries matching exact full NFC casefolded tokens (immutable tuple)."""
            return self._exact_index.get(nfc_tokens, ())

        def lookup_folded(self, folded_tokens: tuple[str, ...]) -> 'tuple[CatalogEntry, ...]':
            """Look up all entries matching full accent-folded tokens (immutable tuple)."""
            return self._folded_index.get(folded_tokens, ())

        def lookup_body_exact(self, nfc_tokens: tuple[str, ...]) -> 'tuple[CatalogEntry, ...]':
            """Look up entries whose body matches exact NFC casefolded tokens (immutable tuple)."""
            return self._body_exact_index.get(nfc_tokens, ())

        def lookup_body_folded(self, folded_tokens: tuple[str, ...]) -> 'tuple[CatalogEntry, ...]':
            """Look up entries whose body matches accent-folded tokens (immutable tuple)."""
            return self._body_folded_index.get(folded_tokens, ())

        def lookup_province_initials(self, text: str) -> 'tuple[CatalogEntry, ...]':
            """Return every province with these body initials; collisions remain explicit."""
            return self._province_initials.get(text, ())

        def window_lengths(self, first_folded_token: str) -> tuple[int, ...]:
            """Only exact windows whose first token can start an indexed name or alias."""
            return self._window_lengths.get(first_folded_token, ())

    # retrieve.py
    FULL_EXACT_SCORE: float = 120.0

    BODY_EXACT_WITH_MARKER_SCORE: float = 115.0

    BARE_EXACT_SCORE: float = 100.0

    FULL_FOLDED_SCORE: float = 70.0

    BODY_FOLDED_WITH_MARKER_SCORE: float = 65.0

    BARE_FOLDED_SCORE: float = 50.0

    MARKER_KINDS = {'phuong': 'phuong', 'p': 'phuong', 'xa': 'xa', 'x': 'xa', 'thi': 'thi_tran', 'tt': 'thi_tran', 'thanh': 'thanh_pho', 'tp': 'thanh_pho', 'tinh': 'tinh', 't': 'tinh', 'duong': 'duong', 'd': 'duong'}

    MARKER_ENDINGS = {'ward': frozenset(('phuong', 'p', 'xa', 'x', 'tt', 'tran')), 'province': frozenset(('pho', 'tp', 'tinh', 't')), 'street': frozenset(('duong', 'd'))}

    MARKER_FIELD_BY_ENDING = {ending: field for field, endings in MARKER_ENDINGS.items() for ending in endings}

    JOINED_MARKER_PATTERNS = {'p': (('ward', '', 'phường', 'phuong'), ('province', 'thành', 'phố', 'pho'), ('ward', '', 'p', 'p')), 'x': (('ward', '', 'xã', 'xa'), ('ward', '', 'x', 'x')), 't': (('ward', 'thị', 'trấn', 'tran'), ('province', '', 'tỉnh', 'tinh'), ('province', '', 'tp', 'tp'), ('ward', '', 'tt', 'tt'), ('province', '', 't', 't')), 'd': (('street', '', 'đường', 'duong'), ('street', '', 'đ', 'd'))}

    @dataclass(frozen=True, slots=True)
    class JoinedMarker:
        field: str
        word: str
        start: int
        body: 'Token'
        exact: bool
        nfc: tuple[str, ...]
        folded: tuple[str, ...]

    @staticmethod
    def joined_markers(tokens: 'list[Token]', index: int, address: str, *, catalog: 'Catalog | None'=None):
        """Yield alternative splits of a token, keeping its literal view available."""
        token = tokens[index]
        if not token.folded:
            return
        recovery_markers = {}
        for field, head, tail, folded_tail in Solution.JOINED_MARKER_PATTERNS.get(token.folded[0], ()):
            if len(token.folded) <= len(folded_tail) or not token.folded.startswith(folded_tail):
                continue
            if catalog is not None and (not catalog.window_lengths(token.folded[len(folded_tail):])):
                continue
            if catalog is None and len(folded_tail) == 1 and (not token.text[0].isupper()):
                continue
            start = token.start
            nfc_head, folded_head = ((), ())
            if head:
                if index == 0:
                    continue
                previous = tokens[index - 1]
                delimiter = address[previous.end:token.start]
                if previous.folded != Solution.fold_accents(head) or '\n' in delimiter or (not Solution.is_valid_marker_delimiter(delimiter)):
                    continue
                start = previous.start
                nfc_head, folded_head = ((previous.nfc,), (previous.folded,))
            cut = 0
            for _ in folded_tail:
                cut += 1
                while cut < len(token.text) and Solution.is_combining_mark(token.text[cut]):
                    cut += 1
                if cut == len(token.text):
                    break
            prefix, suffix = (token.text[:cut], token.text[cut:])
            if not suffix or not suffix[0].isalnum():
                continue
            allows_recovery = len(folded_tail) > 1 or (prefix.isupper() and suffix[0].isupper() and any((c.islower() for c in suffix[1:])))
            if catalog is None and (not allows_recovery) or Solution.fold_accents(prefix) != folded_tail:
                continue
            nfc = Solution.normalize_nfc(suffix).casefold()
            body = Solution.Token(suffix, nfc, Solution.fold_accents(nfc), token.start + cut, token.end)
            prefix_nfc = Solution.normalize_nfc(prefix).casefold()
            marker_nfc = (*nfc_head, prefix_nfc)
            marker_folded = (*folded_head, folded_tail)
            exact = (not head or nfc_head == (head,)) and (prefix_nfc == tail or (tail == 'đ' and prefix_nfc == 'd'))
            marker = Solution.JoinedMarker(field, marker_folded[0], start, body, exact, marker_nfc, marker_folded)
            if catalog is not None:
                yield marker
            else:
                key = (start, field)
                previous = recovery_markers.get(key)
                if previous is None or (marker.exact, body.start) > (previous.exact, previous.body.start):
                    recovery_markers[key] = marker
        yield from recovery_markers.values()

    @staticmethod
    def _retrieve_joined(tokens, catalog, address, literal_candidates):
        literal_scores = {}
        for candidate in literal_candidates:
            key = (candidate.start, candidate.end, candidate.path)
            literal_scores[key] = max(candidate.score, literal_scores.get(key, 0.0))
        for index in range(len(tokens)):
            if tokens[index].folded[:1] not in Solution.JOINED_MARKER_PATTERNS:
                continue
            for marker in Solution.joined_markers(tokens, index, address, catalog=catalog):
                for length in catalog.window_lengths(marker.body.folded):
                    end_index = index + length
                    if end_index > len(tokens):
                        continue
                    window = [marker.body, *tokens[index + 1:end_index]]
                    if any((any((c in address[left.end:right.start] for c in ',;\n')) for left, right in zip(window, window[1:]))):
                        continue
                    end = window[-1].end
                    if window[-1].folded.isdigit() and end_index < len(tokens) and tokens[end_index].folded.isdigit() and address[end:tokens[end_index].start].isspace():
                        continue
                    nfc = tuple((t.nfc for t in window))
                    folded = tuple((t.folded for t in window))
                    exact_ids = {entry.entry_id for entry in catalog.lookup_body_exact(nfc)}
                    for entry in catalog.lookup_body_folded(folded):
                        if entry.field != marker.field or not Solution.accepts_marker(entry, marker.word):
                            continue
                        exact_body = entry.entry_id in exact_ids
                        full = entry.has_prefix and marker.folded == entry.folded_tokens[:len(entry.prefix_tokens)]
                        if full:
                            exact = exact_body and marker.nfc == entry.prefix_tokens
                        else:
                            exact = exact_body and (marker.exact or not entry.has_prefix)
                        path = 'exact' if exact else 'folded'
                        if full:
                            score = (Solution.FULL_EXACT_SCORE if exact else Solution.FULL_FOLDED_SCORE) + 5.0
                        elif exact:
                            score = Solution.BODY_EXACT_WITH_MARKER_SCORE if entry.has_prefix else Solution.FULL_EXACT_SCORE
                        else:
                            score = Solution.BODY_FOLDED_WITH_MARKER_SCORE if entry.has_prefix else Solution.FULL_FOLDED_SCORE
                        literal_score = literal_scores.get((marker.start, end, path))
                        if literal_score is not None:
                            score = min(score, literal_score)
                        if literal_candidates:
                            start_index = index if marker.start == tokens[index].start else index - 1
                            for field in Solution.MARKER_ENDINGS:
                                preceding = Solution.find_preceding_marker(tokens, start_index, field, address)
                                if preceding and (preceding[0], end, path) in literal_scores:
                                    score = min(score, Solution.BARE_EXACT_SCORE if exact else Solution.BARE_FOLDED_SCORE)
                                    break
                        yield Solution.Candidate(entry, marker.start, end, path, score=score, context_marker=True, reasons=('joined_marker', 'marker'))

    @staticmethod
    def accepts_marker(entry: 'CatalogEntry', marker_word: str | None) -> bool:
        """A typed marker cannot support a different canonical administrative type.

        Bare canonical labels have no type to contradict. Both arguments use folded
        tokens, so accented, accentless and abbreviated markers share the same rule.
        """
        if marker_word is None or not entry.has_prefix:
            return True
        return Solution.MARKER_KINDS.get(marker_word) == Solution.MARKER_KINDS.get(entry.folded_tokens[0])

    @staticmethod
    def is_valid_marker_delimiter(sep: str) -> bool:
        """Return True if separation between marker and body is strictly whitespace/dots."""
        return all((c.isspace() or c == '.' for c in sep))

    @staticmethod
    def find_preceding_marker(tokens: 'list[Token]', body_start_idx: int, field: str, address: str) -> tuple[int, bool] | None:
        """Check if token(s) immediately preceding body_start_idx form an administrative marker.

        Returns (marker_start_char_offset, is_exact) or None.
        """
        if body_start_idx < 1:
            return None
        prev_tok = tokens[body_start_idx - 1]
        if Solution.MARKER_FIELD_BY_ENDING.get(prev_tok.folded) != field:
            return None
        sep = address[prev_tok.end:tokens[body_start_idx].start]
        if not Solution.is_valid_marker_delimiter(sep):
            return None
        if body_start_idx >= 2:
            tok2 = tokens[body_start_idx - 2]
            sep2 = address[tok2.end:prev_tok.start]
            if Solution.is_valid_marker_delimiter(sep2):
                if field == 'ward':
                    pair_nfc = (tok2.nfc, prev_tok.nfc)
                    if pair_nfc == ('thị', 'trấn'):
                        return (tok2.start, True)
                    pair_folded = (tok2.folded, prev_tok.folded)
                    if pair_folded == ('thi', 'tran'):
                        return (tok2.start, False)
                elif field == 'province':
                    pair_nfc = (tok2.nfc, prev_tok.nfc)
                    if pair_nfc == ('thành', 'phố'):
                        return (tok2.start, True)
                    pair_folded = (tok2.folded, prev_tok.folded)
                    if pair_folded == ('thanh', 'pho'):
                        return (tok2.start, False)
        if field == 'ward':
            if prev_tok.nfc in ('phường', 'xã', 'p', 'x', 'tt'):
                return (prev_tok.start, True)
            if prev_tok.folded in ('phuong', 'xa'):
                return (prev_tok.start, False)
        elif field == 'province':
            if prev_tok.nfc in ('tỉnh', 'tp', 't'):
                return (prev_tok.start, True)
            if prev_tok.folded in ('tinh',):
                return (prev_tok.start, False)
        elif field == 'street':
            if prev_tok.nfc in ('đường', 'đ', 'd'):
                return (prev_tok.start, True)
            if prev_tok.folded in ('duong',):
                return (prev_tok.start, False)
        return None

    @staticmethod
    def retrieve_candidates(tokens: 'list[Token]', catalog: 'Catalog', config: 'MatcherConfig', address: str) -> 'list[Candidate]':
        """Retrieve all exact and folded window candidates from input tokens.

        Deduplicates candidates on (entry ID, start, end) preferring exact path.
        """
        num_tokens = len(tokens)
        if num_tokens == 0 or catalog.max_tokens == 0:
            return []
        raw_candidates: 'list[Candidate]' = []
        tokens_by_start = {t.start: t for t in tokens}
        token_indices = {t.start: index for index, t in enumerate(tokens)}
        marked_spans_by_field: dict[str, list[tuple[int, int]]] = {'street': [], 'ward': [], 'province': []}
        for i, token in enumerate(tokens):
            for length in catalog.window_lengths(token.folded):
                if i + length > num_tokens:
                    continue
                window = tokens[i:i + length]
                if any((any((c in address[left.end:right.start] for c in ',;\n')) for left, right in zip(window, window[1:]))):
                    continue
                body_start_char = window[0].start
                body_end_char = window[-1].end
                window_nfc = tuple((t.nfc for t in window))
                window_folded = tuple((t.folded for t in window))
                if length == 1:
                    marker_info = Solution.find_preceding_marker(tokens, i, 'province', address)
                    for entry in catalog.lookup_province_initials(window_folded[0]):
                        marker_word = tokens_by_start[marker_info[0]].folded if marker_info else None
                        if not Solution.accepts_marker(entry, marker_word):
                            continue
                        raw_candidates.append(Solution.Candidate(entry=entry, start=marker_info[0] if marker_info else body_start_char, end=body_end_char, path='folded', score=60.0 if marker_info else 45.0, context_marker=marker_info is not None, reasons=('province_initials',)))
                if window[-1].folded.isdigit() and i + length < num_tokens and tokens[i + length].folded.isdigit() and address[window[-1].end:tokens[i + length].start].isspace():
                    continue
                exact_full = catalog.lookup_exact(window_nfc)
                exact_full_ids = set()
                for entry in exact_full:
                    exact_full_ids.add(entry.entry_id)
                    has_marker = entry.has_prefix
                    cand_start = body_start_char
                    if not has_marker:
                        marker_info = Solution.find_preceding_marker(tokens, i, entry.field, address)
                        if marker_info is not None:
                            has_marker = True
                            cand_start = marker_info[0]
                    if entry.field != 'street' and (not has_marker) and all((t.replace('/', '').isdigit() for t in window_folded)):
                        continue
                    score = (Solution.FULL_EXACT_SCORE if has_marker else Solution.BARE_EXACT_SCORE) + (5.0 if entry.has_prefix else 0.0)
                    reasons = ('exact_full',) + (('marker',) if has_marker else ())
                    raw_candidates.append(Solution.Candidate(entry=entry, start=cand_start, end=body_end_char, path='exact', cost=0, score=score, context_marker=has_marker, reasons=reasons))
                    if has_marker:
                        marked_spans_by_field[entry.field].append((cand_start, body_end_char))
                folded_full = catalog.lookup_folded(window_folded)
                for entry in folded_full:
                    if entry.entry_id in exact_full_ids:
                        continue
                    has_marker = entry.has_prefix
                    cand_start = body_start_char
                    if not has_marker:
                        marker_info = Solution.find_preceding_marker(tokens, i, entry.field, address)
                        if marker_info is not None:
                            has_marker = True
                            cand_start = marker_info[0]
                    if entry.field != 'street' and (not has_marker) and all((t.replace('/', '').isdigit() for t in window_folded)):
                        continue
                    score = (Solution.FULL_FOLDED_SCORE if has_marker else Solution.BARE_FOLDED_SCORE) + (5.0 if entry.has_prefix else 0.0)
                    reasons = ('folded_full',) + (('marker',) if has_marker else ())
                    raw_candidates.append(Solution.Candidate(entry=entry, start=cand_start, end=body_end_char, path='folded', cost=0, score=score, context_marker=has_marker, reasons=reasons))
                exact_body = catalog.lookup_body_exact(window_nfc)
                exact_body_ids = set()
                for entry in exact_body:
                    if not entry.has_prefix:
                        continue
                    exact_body_ids.add(entry.entry_id)
                    marker_info = Solution.find_preceding_marker(tokens, i, entry.field, address)
                    if marker_info is not None and (not Solution.accepts_marker(entry, tokens_by_start[marker_info[0]].folded)):
                        continue
                    has_marker = marker_info is not None
                    if not has_marker and all((t == 'so' or t.replace('/', '').isdigit() for t in window_folded)):
                        continue
                    cand_start = marker_info[0] if marker_info is not None else body_start_char
                    marker_exact = marker_info[1] if marker_info is not None else True
                    path = 'exact' if marker_exact else 'folded'
                    if path == 'exact':
                        score = Solution.BODY_EXACT_WITH_MARKER_SCORE if has_marker else Solution.BARE_EXACT_SCORE
                    else:
                        score = Solution.BODY_FOLDED_WITH_MARKER_SCORE if has_marker else Solution.BARE_FOLDED_SCORE
                    reasons = ('exact_body',) + (('marker',) if has_marker else ())
                    raw_candidates.append(Solution.Candidate(entry=entry, start=cand_start, end=body_end_char, path=path, cost=0, score=score, context_marker=has_marker, reasons=reasons))
                    if has_marker:
                        marked_spans_by_field[entry.field].append((cand_start, body_end_char))
                folded_body = catalog.lookup_body_folded(window_folded)
                for entry in folded_body:
                    if not entry.has_prefix or entry.entry_id in exact_body_ids:
                        continue
                    marker_info = Solution.find_preceding_marker(tokens, i, entry.field, address)
                    if marker_info is not None and (not Solution.accepts_marker(entry, tokens_by_start[marker_info[0]].folded)):
                        continue
                    has_marker = marker_info is not None
                    if not has_marker and all((t == 'so' or t.replace('/', '').isdigit() for t in window_folded)):
                        continue
                    cand_start = marker_info[0] if marker_info is not None else body_start_char
                    score = Solution.BODY_FOLDED_WITH_MARKER_SCORE if has_marker else Solution.BARE_FOLDED_SCORE
                    reasons = ('folded_body',) + (('marker',) if has_marker else ())
                    raw_candidates.append(Solution.Candidate(entry=entry, start=cand_start, end=body_end_char, path='folded', cost=0, score=score, context_marker=has_marker, reasons=reasons))
        for candidate in Solution._retrieve_joined(tokens, catalog, address, raw_candidates):
            raw_candidates.append(candidate)
            if candidate.path == 'exact':
                marked_spans_by_field[candidate.field].append(candidate.span)
        enhanced_candidates: 'list[Candidate]' = []
        marked_street_ends = {end for _, end in marked_spans_by_field['street']}
        for cand in raw_candidates:
            if not cand.context_marker and cand.entry.field == 'street':
                start_idx = token_indices.get(cand.start)
                preceding_tok_idx = start_idx - 1 if start_idx is not None else None
                if preceding_tok_idx is not None and preceding_tok_idx >= 1:
                    conj_tok = tokens[preceding_tok_idx]
                    if conj_tok.nfc in ('và', 'va'):
                        m_end = tokens[preceding_tok_idx - 1].end
                        if m_end in marked_street_ends and address[m_end:conj_tok.start].isspace():
                            score = Solution.FULL_EXACT_SCORE if cand.path == 'exact' else Solution.FULL_FOLDED_SCORE
                            cand = Solution.Candidate(entry=cand.entry, start=cand.start, end=cand.end, path=cand.path, cost=cand.cost, score=score, context_marker=True, reasons=cand.reasons + ('conjunction_marker',))
            enhanced_candidates.append(cand)
        deduped: 'dict[tuple[int, int, int], Candidate]' = {}
        for cand in enhanced_candidates:
            key = (cand.entry.entry_id, cand.start, cand.end)
            if key not in deduped:
                deduped[key] = cand
            else:
                existing = deduped[key]
                if existing.path == 'folded' and cand.path == 'exact':
                    deduped[key] = cand
                elif existing.path == cand.path:
                    if cand.score > existing.score:
                        deduped[key] = cand
        return sorted(deduped.values(), key=lambda c: (-c.score, c.cost, c.start, c.end, c.entry.entry_id))

    # verify.py
    @staticmethod
    def thresholded_osa_distance(s1: str, s2: str, limit: int) -> int:
        """Compute Optimal String Alignment distance between s1 and s2 bounded by limit.

        If the edit distance exceeds limit, returns limit + 1 early.
        Employs early length rejection, banded traversal (|i - j| <= limit),
        and row-minimum pruning.

        Args:
            s1: First string.
            s2: Second string.
            limit: Maximum allowable edit distance (non-negative integer).

        Returns:
            Exact distance if <= limit, otherwise limit + 1.
        """
        if limit < 0:
            return limit + 1
        if s1 == s2:
            return 0
        n = len(s1)
        m = len(s2)
        if abs(n - m) > limit:
            return limit + 1
        if n == 0:
            return m if m <= limit else limit + 1
        if m == 0:
            return n if n <= limit else limit + 1
        infinity = limit + 1
        k = limit
        row_prev2 = [infinity] * (m + 1)
        row_prev = [infinity] * (m + 1)
        row_curr = [infinity] * (m + 1)
        for j in range(min(m + 1, k + 1)):
            row_prev[j] = j
        for i in range(1, n + 1):
            row_curr = [infinity] * (m + 1)
            j_min = max(1, i - k)
            j_max = min(m, i + k)
            if i <= k:
                row_curr[0] = i
                row_min = i
            else:
                row_min = infinity
            s1_char = s1[i - 1]
            s1_prev_char = s1[i - 2] if i > 1 else None
            for j in range(j_min, j_max + 1):
                s2_char = s2[j - 1]
                cost = 0 if s1_char == s2_char else 1
                del_cost = row_prev[j] + 1
                ins_cost = row_curr[j - 1] + 1
                sub_cost = row_prev[j - 1] + cost
                val = del_cost if del_cost < ins_cost else ins_cost
                if sub_cost < val:
                    val = sub_cost
                if i > 1 and j > 1:
                    if s1_char == s2[j - 2] and s1_prev_char == s2_char:
                        trans_cost = row_prev2[j - 2] + 1
                        if trans_cost < val:
                            val = trans_cost
                if val > infinity:
                    val = infinity
                row_curr[j] = val
                if val < row_min:
                    row_min = val
            if row_min > k:
                return infinity
            row_prev2 = row_prev
            row_prev = row_curr
        res = row_curr[m]
        return res if res <= k else infinity

    # select.py
    FIELDS: tuple[str, ...] = ('street', 'ward', 'province')

    @staticmethod
    def supersede_contained_prefixes(candidates: 'list[Candidate]') -> 'list[Candidate]':
        """Prefer complete names over their contained fragments at one occurrence.

        Full exact/folded names explain unmarked fragments across fields; recovered
        names can extend a prefix within the same field. Separate occurrences and
        explicitly marked competing fields remain independent.
        """
        by_start: 'dict[int, list[Candidate]]' = {}
        for candidate in candidates:
            by_start.setdefault(candidate.start, []).append(candidate)
        starts = sorted(by_start)
        removed = set()
        for longer in candidates:
            for pos in range(bisect_left(starts, longer.start), bisect_left(starts, longer.end)):
                for short in by_start[starts[pos]]:
                    if short is longer or short.end > longer.end or (short.start == longer.start and short.end == longer.end) or (short.field == longer.field and short.label == longer.label):
                        continue
                    if short.field != longer.field and short.context_marker:
                        continue
                    if longer.path in ('exact', 'folded') and 'province_initials' not in longer.reasons:
                        attr = 'body_nfc_tokens' if short.path == 'exact' else 'body_folded_tokens'
                        small, large = (getattr(short.entry, attr), getattr(longer.entry, attr))
                        if small and len(small) < len(large) and any((large[i:i + len(small)] == small for i in range(len(large) - len(small) + 1))):
                            removed.add(id(short))
                    elif short.field == longer.field and short.start == longer.start and (longer.path in ('fuzzy', 'compact', 'telex', 'vni')) and (longer.cost <= 2) and (longer.score >= 10.0):
                        for attr in ('nfc_tokens', 'body_nfc_tokens', 'folded_tokens', 'body_folded_tokens'):
                            small, large = (getattr(short.entry, attr), getattr(longer.entry, attr))
                            if small and len(small) < len(large) and (large[:len(small)] == small):
                                removed.add(id(short))
                                break
        return [c for c in candidates if id(c) not in removed]

    @staticmethod
    def spans_overlap(c1: 'Candidate', c2: 'Candidate') -> bool:
        """Return True if character spans of two candidates overlap."""
        return max(c1.start, c2.start) < min(c1.end, c2.end)

    @staticmethod
    def select_candidates(verified_candidates: 'list[Candidate]', config: 'MatcherConfig') -> 'tuple[dict[str, str | None], list[Candidate], dict[str, Any]]':
        """Jointly evaluate candidate combinations and return (result, retained_candidates, selection_trace).

        - result: Fresh dictionary with exact keys 'street', 'ward', 'province'.
        - retained_candidates: List of post-pruning Candidate objects evaluated in joint selection.
        - selection_trace: JSON-serializable dictionary with best_score, ambiguous_fields,
          pruned_fields, and rejection_reasons.
        """
        post_supersede = Solution.supersede_contained_prefixes([c for c in verified_candidates if c.score >= config.min_score])
        by_field: 'dict[str, list[Candidate]]' = {f: [] for f in Solution.FIELDS}
        for c in post_supersede:
            by_field[c.field].append(c)
        pruned_fields: list[str] = []
        retained_by_field: 'dict[str, list[Candidate]]' = {}
        for f in Solution.FIELDS:
            cands_f = by_field[f]
            sorted_f = sorted(cands_f, key=lambda c: (-c.score, c.cost, c.start, c.end, c.entry.entry_id))
            if len(sorted_f) > config.top_k:
                pruned_fields.append(f)
                retained_by_field[f] = []
            else:
                retained_by_field[f] = sorted_f
        retained_candidates: 'list[Candidate]' = []
        for f in Solution.FIELDS:
            retained_candidates.extend(retained_by_field[f])
        choices: 'dict[str, list[Candidate | None]]' = {f: list(retained_by_field[f]) + [None] for f in Solution.FIELDS}
        best_score: float = 0.0
        label_max_score: dict[str, dict[str | None, float]] = {f: {} for f in Solution.FIELDS}
        for cs in choices['street']:
            for cw in choices['ward']:
                if cs is not None and cw is not None and Solution.spans_overlap(cs, cw):
                    continue
                for cp in choices['province']:
                    if cp is not None:
                        if cs is not None and Solution.spans_overlap(cs, cp):
                            continue
                        if cw is not None and Solution.spans_overlap(cw, cp):
                            continue
                    combo_score = (cs.score if cs else 0.0) + (cw.score if cw else 0.0) + (cp.score if cp else 0.0)
                    if combo_score > best_score:
                        best_score = combo_score
                    for f, cand in (('street', cs), ('ward', cw), ('province', cp)):
                        lbl = cand.label if cand is not None else None
                        prev_max = label_max_score[f].get(lbl, -1.0)
                        if combo_score > prev_max:
                            label_max_score[f][lbl] = combo_score
        threshold = best_score - config.ambiguity_margin
        ambiguous_fields: list[str] = []
        rejection_reasons: dict[str, str] = {}
        result: dict[str, str | None] = {'street': None, 'ward': None, 'province': None}
        for f in Solution.FIELDS:
            if f in pruned_fields:
                result[f] = None
                rejection_reasons[f] = 'pruned'
                continue
            if not retained_by_field[f]:
                result[f] = None
                rejection_reasons[f] = 'no_candidates'
                continue
            labels_in_margin = [lbl for lbl, max_s in label_max_score[f].items() if max_s >= threshold - 1e-09]
            non_null_labels = [lbl for lbl in labels_in_margin if lbl is not None]
            has_null = None in labels_in_margin
            if not has_null and len(non_null_labels) == 1:
                winning_label = non_null_labels[0]
                best_cand_score = max((c.score for c in retained_by_field[f] if c.label == winning_label), default=0.0)
                if best_cand_score >= config.min_score:
                    result[f] = winning_label
                else:
                    result[f] = None
                    rejection_reasons[f] = 'below_min_score'
            else:
                result[f] = None
                ambiguous_fields.append(f)
                rejection_reasons[f] = 'ambiguous'
        selection_trace: dict[str, Any] = {'best_score': round(float(best_score), 4), 'ambiguous_fields': sorted(ambiguous_fields), 'pruned_fields': sorted(pruned_fields), 'rejection_reasons': rejection_reasons}
        return (result, retained_candidates, selection_trace)

    # approximate.py
    COMPONENT_SEPARATOR = re.compile('[,;\\n]')

    class ExhaustedDict(dict):
        """Dictionary of budget exhaustion flags that evaluates truthy if any flag is True."""

        def __bool__(self) -> bool:
            return any(self.values())

    @dataclass(frozen=True, slots=True)
    class TargetForm:
        """An indexed catalog target form (full canonical or body only)."""
        form_id: int
        entry: 'CatalogEntry'
        form_type: str
        folded_text: str
        length: int
        trigrams: tuple[str, ...]

    @dataclass(frozen=True, slots=True)
    class CandidateSpan:
        """A candidate text window in the source address for approximate matching."""
        start: int
        end: int
        words: 'tuple[Token, ...]'
        folded_text: str
        has_marker: bool
        field_hint: str | None
        is_extension_of: 'Candidate | None'
        marker_word: str | None = None
        joined: bool = False
        literal_alternatives: tuple[str, ...] = ()

    @staticmethod
    def extract_trigrams(text: str) -> tuple[str, ...]:
        """Extract character trigrams from normalized string."""
        if len(text) < 3:
            return ()
        return tuple((text[i:i + 3] for i in range(len(text) - 2)))

    class ApproximateIndex:
        """Immutable inverted trigram index and length buckets for catalog forms."""

        def __init__(self, catalog: 'Catalog') -> None:
            forms: 'list[TargetForm]' = []
            postings: 'dict[str, list[TargetForm]]' = {}
            length_buckets: 'dict[int, list[TargetForm]]' = {}
            short_name_forms: 'list[TargetForm]' = []
            form_id = 0
            for entry in catalog.entries:
                full_folded = ' '.join(entry.folded_tokens)
                if not any((c.isdigit() for c in full_folded)) and len(full_folded) >= 5:
                    full_grams = Solution.extract_trigrams(full_folded)
                    tf_full = Solution.TargetForm(form_id=form_id, entry=entry, form_type='full', folded_text=full_folded, length=len(full_folded), trigrams=full_grams)
                    forms.append(tf_full)
                    form_id += 1
                    length_buckets.setdefault(tf_full.length, []).append(tf_full)
                    if 5 <= tf_full.length <= 7:
                        short_name_forms.append(tf_full)
                    for g in set(full_grams):
                        postings.setdefault(g, []).append(tf_full)
                if entry.has_prefix:
                    body_folded = ' '.join(entry.body_folded_tokens)
                    if not any((c.isdigit() for c in body_folded)) and len(body_folded) >= 5:
                        if body_folded != full_folded:
                            body_grams = Solution.extract_trigrams(body_folded)
                            tf_body = Solution.TargetForm(form_id=form_id, entry=entry, form_type='body', folded_text=body_folded, length=len(body_folded), trigrams=body_grams)
                            forms.append(tf_body)
                            form_id += 1
                            length_buckets.setdefault(tf_body.length, []).append(tf_body)
                            if 5 <= tf_body.length <= 7:
                                short_name_forms.append(tf_body)
                            for g in set(body_grams):
                                postings.setdefault(g, []).append(tf_body)
            self._forms: 'tuple[TargetForm, ...]' = tuple(forms)
            self._postings: 'dict[str, tuple[TargetForm, ...]]' = {k: tuple(v) for k, v in postings.items()}
            self._length_buckets: 'dict[int, tuple[TargetForm, ...]]' = {k: tuple(v) for k, v in length_buckets.items()}
            self._short_name_forms: 'tuple[TargetForm, ...]' = tuple(short_name_forms)
            self.max_length = max(length_buckets, default=0)
            self._gram_frequency: dict[str, int] = {g: len(p) for g, p in self._postings.items()}

        @property
        def gram_frequency(self) -> dict[str, int]:
            return self._gram_frequency

        @property
        def postings(self) -> 'dict[str, tuple[TargetForm, ...]]':
            return self._postings

        @property
        def short_name_forms(self) -> 'tuple[TargetForm, ...]':
            return self._short_name_forms

        @property
        def length_buckets(self) -> 'dict[int, tuple[TargetForm, ...]]':
            return self._length_buckets

    @staticmethod
    def extract_spans(address: str, tokens: 'list[Token]', exact_candidates: 'list[Candidate]', max_spans: int) -> 'tuple[list[CandidateSpan], bool, list[Candidate]]':
        """Extract bounded component windows and non-greedy extension spans.

        Returns:
            (spans, spans_exhausted, suspicious_exact_candidates)
        """
        spans = []
        seen = set()
        suspicious = []
        exhausted = False
        coverage = sorted(((c.start, c.end) for c in exact_candidates))
        coverage_position, covered_end = (0, -1)
        candidates_by_start = {}
        for candidate in exact_candidates:
            candidates_by_start.setdefault(candidate.start, []).append(candidate)

        def add_span(start, words, field, prefix, marker_word, *, joined=False, literal_alternatives=()):
            nonlocal exhausted
            end = words[-1].end
            folded = ' '.join((word.folded for word in words))
            key = (start, end, folded, field)
            if key in seen:
                return
            if len(spans) >= max_spans:
                exhausted = True
                return
            seen.add(key)
            spans.append(Solution.CandidateSpan(start, end, words, folded, field is not None, field, prefix, marker_word, joined, literal_alternatives))
        boundaries = [0]
        for index in range(1, len(tokens)):
            if Solution.COMPONENT_SEPARATOR.search(address, tokens[index - 1].end, tokens[index].start):
                boundaries.append(index)
        boundaries.append(len(tokens))
        candidate_starts = sorted(candidates_by_start)
        for group_start, group_end in zip(boundaries, boundaries[1:]):
            if exhausted:
                exact_pos = bisect_left(candidate_starts, tokens[group_start].start)
                if exact_pos == len(candidate_starts) or candidate_starts[exact_pos] >= tokens[group_end - 1].end:
                    continue
            markers = {}
            for index in range(group_start + 1, group_end):
                previous_token = tokens[index - 1]
                field = Solution.MARKER_FIELD_BY_ENDING.get(previous_token.folded)
                if field is not None:
                    marker = Solution.find_preceding_marker(tokens, index, field, address)
                    if marker is not None:
                        marker_index = index - 1 if previous_token.start == marker[0] else index - 2
                        if marker_index >= group_start:
                            markers[marker_index] = (index, field)
            cuts = sorted({group_start, group_end, *markers})
            for left, right in zip(cuts, cuts[1:]):
                hint = markers.get(left)
                body = hint[0] if hint else left
                if hint is None:
                    while body < right and (tokens[body].folded.replace('/', '').isdigit() or (tokens[body].folded == 'so' and body + 1 < right and tokens[body + 1].folded.replace('/', '').isdigit())):
                        body += 1
                    left = body
                if body >= right:
                    continue
                start, end = (tokens[left].start, tokens[right - 1].end)
                while coverage_position < len(coverage) and coverage[coverage_position][0] <= start:
                    covered_end = max(covered_end, coverage[coverage_position][1])
                    coverage_position += 1
                if covered_end >= end:
                    continue
                prefix = next((c for c in candidates_by_start.get(start, ()) if c.end < end and (hint is None or c.field == hint[1])), None)
                if prefix is not None:
                    suspicious.append(prefix)
                alternatives = {}
                for pos in range(body, right):
                    if tokens[pos].folded[:1] not in Solution.JOINED_MARKER_PATTERNS:
                        continue
                    for marker in Solution.joined_markers(tokens, pos, address):
                        marker_pos = pos if marker.start == tokens[pos].start else pos - 1
                        if marker_pos >= body:
                            alternatives.setdefault(marker_pos, []).append((pos, marker))
                for offset in dict.fromkeys((left, body)):
                    words = tuple(tokens[offset:right])
                    split_forms = tuple((' '.join((marker.body.folded, *(token.folded for token in tokens[pos + 1:right]))) for pos, marker in alternatives.get(offset, ())))
                    add_span(start, words, hint[1] if hint else None, prefix, tokens[left].folded if hint else None, literal_alternatives=split_forms)
                joined_cuts = [*sorted(alternatives), right]
                for marker_pos, stop in zip(joined_cuts, joined_cuts[1:]):
                    for pos, marker in alternatives[marker_pos]:
                        if pos >= stop:
                            continue
                        joined_end = tokens[stop - 1].end
                        matching = [c for c in candidates_by_start.get(marker.start, ()) if c.field == marker.field]
                        if any((c.end >= joined_end for c in matching)):
                            continue
                        joined_prefix = next(iter(matching), None)
                        if joined_prefix is not None:
                            suspicious.append(joined_prefix)
                        words = (marker.body, *tokens[pos + 1:stop])
                        add_span(marker.start, words, marker.field, joined_prefix, marker.word, joined=True)
        spans.sort(key=lambda span: (span.is_extension_of is None, span.start, -(span.end - span.start), span.folded_text))
        return (spans, exhausted, suspicious)

    @staticmethod
    def recover_fuzzy_candidates(span_data: 'tuple[list[CandidateSpan], bool, list[Candidate]]', config: 'MatcherConfig', approx_index: 'ApproximateIndex') -> 'tuple[list[Candidate], list[Candidate], dict[str, Any], set[str], list[Candidate]]':
        """Execute bounded approximate recovery using trigrams and thresholded OSA.

        Returns:
            (retrieved_fuzzy, verified_fuzzy, budget_trace, exhausted_fields, suspicious_exact_to_abstain)
        """
        postings_count = 0
        verifications_count = 0
        max_cands_in_span = 0
        postings_exhausted = False
        verifications_exhausted = False
        candidates_per_span_exhausted = False
        exhausted_fields: set[str] = set()
        spans, spans_exhausted, suspicious_exact = span_data
        if spans_exhausted:
            exhausted_fields.update({'street', 'ward', 'province'})
        retrieved_fuzzy: 'list[Candidate]' = []
        verified_fuzzy: 'list[Candidate]' = []
        verify_cache: dict[tuple[str, str, int], int] = {}
        for span in spans:
            span_folded = span.folded_text
            span_len = len(span_folded)
            if span_len < 5 or span_len > approx_index.max_length + config.max_edits or any((c.isdigit() for c in span_folded)):
                continue
            span_grams = Solution.extract_trigrams(span_folded)
            unique_grams = sorted((g for g in set(span_grams) if g in approx_index.gram_frequency), key=lambda g: (approx_index.gram_frequency[g], g))[:3]
            form_gram_hits: 'dict[TargetForm, int]' = {}
            for g in unique_grams:
                postings_list = approx_index.postings.get(g, ())
                for target_form in postings_list:
                    if postings_count >= config.max_postings:
                        postings_exhausted = True
                        exhausted_fields.update((span.field_hint,) if span.field_hint else ('street', 'ward', 'province'))
                        break
                    postings_count += 1
                    if abs(target_form.length - span_len) <= config.max_edits and (span.field_hint is None or target_form.entry.field == span.field_hint) and (not span.joined or not target_form.entry.has_prefix or target_form.form_type == 'body') and Solution.accepts_marker(target_form.entry, span.marker_word):
                        form_gram_hits[target_form] = form_gram_hits.get(target_form, 0) + 1
                if postings_exhausted:
                    break
            if span_len <= 7 and (not postings_exhausted):
                for tf in approx_index.short_name_forms:
                    if postings_count >= config.max_postings:
                        postings_exhausted = True
                        exhausted_fields.update((span.field_hint,) if span.field_hint else ('street', 'ward', 'province'))
                        break
                    postings_count += 1
                    if abs(tf.length - span_len) <= config.max_edits and (span.field_hint is None or tf.entry.field == span.field_hint) and (not span.joined or not tf.entry.has_prefix or tf.form_type == 'body') and Solution.accepts_marker(tf.entry, span.marker_word):
                        form_gram_hits.setdefault(tf, 0)
            if not form_gram_hits:
                continue
            sorted_forms = sorted(form_gram_hits.keys(), key=lambda f: (-form_gram_hits[f], abs(f.length - span_len), f.entry.entry_id, f.form_type))
            cand_count = len(sorted_forms)
            if cand_count > config.max_candidates_per_span:
                candidates_per_span_exhausted = True
                exhausted_fields.update((form.entry.field for form in sorted_forms))
                sorted_forms = sorted_forms[:config.max_candidates_per_span]
            retained_count = len(sorted_forms)
            if retained_count > max_cands_in_span:
                max_cands_in_span = retained_count
            span_retrieved: 'list[Candidate]' = []
            for form in sorted_forms:
                cand = Solution.Candidate(entry=form.entry, start=span.start, end=span.end, path='fuzzy', cost=0, score=0.0, context_marker=span.has_marker, reasons=('retrieved_fuzzy',))
                span_retrieved.append(cand)
                retrieved_fuzzy.append(cand)
            for form in sorted_forms:
                if verifications_count >= config.max_verifications:
                    verifications_exhausted = True
                    exhausted_fields.update((span.field_hint,) if span.field_hint else ('street', 'ward', 'province'))
                    break
                target_len = form.length
                allowed_edits = 1 if target_len < 9 or span_len < 9 else min(2, config.max_edits)
                cache_key = (span_folded, form.folded_text, allowed_edits)
                if cache_key in verify_cache:
                    dist = verify_cache[cache_key]
                else:
                    verifications_count += 1
                    dist = Solution.thresholded_osa_distance(span_folded, form.folded_text, limit=allowed_edits)
                    verify_cache[cache_key] = dist
                if 0 < dist <= allowed_edits and max(target_len, span_len) - dist >= 4:
                    closer_split = False
                    for alternative in span.literal_alternatives:
                        if abs(len(alternative) - target_len) >= dist:
                            continue
                        if alternative == form.folded_text:
                            closer_split = True
                            break
                        if verifications_count >= config.max_verifications:
                            verifications_exhausted = True
                            exhausted_fields.add(form.entry.field)
                            break
                        verifications_count += 1
                        if Solution.thresholded_osa_distance(alternative, form.folded_text, dist - 1) < dist:
                            closer_split = True
                            break
                    if closer_split or verifications_exhausted:
                        continue
                    has_marker = span.has_marker or (form.form_type == 'full' and form.entry.has_prefix)
                    if form.form_type == 'full':
                        base_score = (70.0 if has_marker else 50.0) + (5.0 if form.entry.has_prefix else 0.0)
                    else:
                        base_score = 65.0 if has_marker else 50.0
                    score = base_score - dist * 10.0
                    reasons = ('fuzzy_verified', f'dist_{dist}') + (('marker',) if has_marker else ())
                    if span.is_extension_of is not None:
                        reasons = reasons + ('extension_recovery',)
                    v_cand = Solution.Candidate(entry=form.entry, start=span.start, end=span.end, path='fuzzy', cost=dist, score=score, context_marker=has_marker, reasons=reasons, body_start=span.words[0].start if span.joined else None)
                    verified_fuzzy.append(v_cand)
            if verifications_exhausted:
                break
        any_exhausted = bool(spans_exhausted or postings_exhausted or verifications_exhausted or candidates_per_span_exhausted)
        exhausted_map = Solution.ExhaustedDict({'fuzzy_spans': spans_exhausted, 'postings': postings_exhausted, 'verifications': verifications_exhausted, 'candidates_per_span': candidates_per_span_exhausted})
        budget_trace: dict[str, Any] = {'counts': {'fuzzy_spans': len(spans), 'postings': postings_count, 'verifications': verifications_count, 'candidates_per_span': max_cands_in_span}, 'limits': {'fuzzy_spans': config.max_fuzzy_spans, 'postings': config.max_postings, 'verifications': config.max_verifications, 'candidates_per_span': config.max_candidates_per_span}, 'exhausted': any_exhausted, 'exhaustion_flags': dict(exhausted_map)}
        suspicious_to_abstain: 'list[Candidate]' = []
        if any_exhausted:
            suspicious_to_abstain = [c for c in suspicious_exact if c.field in exhausted_fields]
        return (retrieved_fuzzy, verified_fuzzy, budget_trace, exhausted_fields, suspicious_to_abstain)

    # typing.py
    TONES = dict(zip('sfrxj', '̣́̀̉̃'))

    VNI_TONES = dict(zip('12345', '̣́̀̉̃'))

    SHAPES = {'telex': {'dd': 'đ', 'aa': 'â', 'aw': 'ă', 'ee': 'ê', 'oo': 'ô', 'ow': 'ơ', 'uw': 'ư'}, 'vni': {'d9': 'đ', 'a6': 'â', 'a8': 'ă', 'e6': 'ê', 'o6': 'ô', 'o7': 'ơ', 'u7': 'ư'}}

    @dataclass(frozen=True)
    class Hypothesis:
        text: str
        tone: str | None
        changed: bool

    @staticmethod
    def decode_token(text: str, mode: str) -> 'Hypothesis':
        text = unicodedata.normalize('NFC', text).casefold()
        changed = False
        tone = None
        table = Solution.TONES if mode == 'telex' else Solution.VNI_TONES
        if len(text) > 2 and text[-1] in table and any((c in Solution.fold_accents(text[:-1]) for c in 'aeiouy')):
            tone = table[text[-1]]
            text = text[:-1]
            changed = True
        for raw, composed in Solution.SHAPES[mode].items():
            if raw in text:
                text = text.replace(raw, composed)
                changed = True
        return Solution.Hypothesis(text, tone, changed)

    @staticmethod
    def compatible(hypothesis: 'Hypothesis', target: str) -> bool:
        observed = unicodedata.normalize('NFD', hypothesis.text)
        target_nfd = unicodedata.normalize('NFD', target)
        tones = set(Solution.TONES.values())
        required = {c for c in observed if c in tones}
        if hypothesis.tone:
            required.add(hypothesis.tone)
        if required and required != {c for c in target_nfd if c in tones}:
            return False
        for shape in ('̂', '̆', '̛', 'đ'):
            if observed.count(shape) > target_nfd.count(shape):
                return False
        return Solution.fold_accents(hypothesis.text) == Solution.fold_accents(target)

    @staticmethod
    def preserves_compact_accents(observed: str, target: str) -> bool:
        """Keep explicit accents at their character positions when spaces change."""

        def clusters(text: str) -> list[str]:
            result = []
            for char in unicodedata.normalize('NFD', text):
                if unicodedata.combining(char) and result:
                    result[-1] += char
                else:
                    result.append(char)
            return result
        left, right = (clusters(observed), clusters(target))
        return len(left) == len(right) and all((Solution.fold_accents(a) == Solution.fold_accents(b) and set(a[1:]) <= set(b[1:]) and (a[0] != 'đ' or b[0] == 'đ') for a, b in zip(left, right)))

    class TypingIndex:

        def __init__(self, catalog: 'Catalog') -> None:
            self.compact = {}
            self.words = {}
            self.max_length = 0
            for entry in catalog.entries:
                forms = {entry.nfc_tokens: entry.folded_tokens, entry.body_nfc_tokens: entry.body_folded_tokens}
                for words, folded_words in forms.items():
                    if not words:
                        continue
                    key = ''.join(folded_words)
                    if len(key) < 4 or not key.isalpha():
                        continue
                    self.max_length = max(self.max_length, len(key))
                    record = (entry, words)
                    self.compact.setdefault(key, []).append(record)
                    self.words.setdefault(folded_words, []).append(record)

    @staticmethod
    def recover_typing(address: str, span_data: 'tuple[list[CandidateSpan], bool, list[Candidate]]', index: 'TypingIndex', config: 'MatcherConfig') -> 'tuple[list[Candidate], list[Candidate], dict[str, Any], set[str], list[Candidate]]':
        """Return retrieved, verified, budgets, exhausted fields, suspicious prefixes."""
        spans, exhausted, suspicious = span_data
        counts = dict(fuzzy_spans=len(spans), postings=0, verifications=0, candidates_per_span=0)
        limits = dict(fuzzy_spans=config.max_fuzzy_spans, postings=config.max_postings, verifications=config.max_verifications, candidates_per_span=config.max_candidates_per_span)
        flags = {key: False for key in limits}
        flags['fuzzy_spans'] = exhausted
        fields = set(('street', 'ward', 'province')) if exhausted else set()
        retrieved, verified = ([], [])
        for span in spans:
            if len(span.folded_text) > 2 * index.max_length + 2 * len(span.words) - 1:
                continue
            words = [token.nfc for token in span.words]
            modes = []
            compact = ''.join((token.folded for token in span.words))
            if config.enable_compact and 5 <= len(compact) <= index.max_length:
                modes.append(('compact', index.compact.get(compact, ()), None))
            if config.enable_typing:
                for mode in ('telex', 'vni'):
                    hypotheses = [Solution.decode_token(w, mode) for w in words]
                    if any((h.changed for h in hypotheses)):
                        key = tuple((Solution.fold_accents(h.text) for h in hypotheses))
                        modes.append((mode, index.words.get(key, ()), hypotheses))
            span_visited = 0
            for mode, records, hypotheses in modes:
                for entry, target in records:
                    if counts['postings'] >= limits['postings'] or span_visited >= limits['candidates_per_span']:
                        flags['postings' if counts['postings'] >= limits['postings'] else 'candidates_per_span'] = True
                        fields.update((span.field_hint,) if span.field_hint else ('street', 'ward', 'province'))
                        break
                    counts['postings'] += 1
                    span_visited += 1
                    counts['candidates_per_span'] = max(counts['candidates_per_span'], span_visited)
                    if span.field_hint and entry.field != span.field_hint:
                        continue
                    if not Solution.accepts_marker(entry, span.marker_word):
                        continue
                    if span.joined and target != entry.body_nfc_tokens:
                        continue
                    candidate = Solution.Candidate(entry, span.start, span.end, mode, score=0)
                    retrieved.append(candidate)
                    if counts['verifications'] >= limits['verifications']:
                        flags['verifications'] = True
                        fields.add(entry.field)
                        continue
                    counts['verifications'] += 1
                    if mode == 'compact':
                        source = address[span.words[0].start:span.end]
                        if any((not (c.isalpha() or c.isspace() or unicodedata.category(c).startswith('M')) for c in source)):
                            continue
                        if tuple((Solution.fold_accents(w) for w in words)) == tuple((Solution.fold_accents(w) for w in target)):
                            continue
                        if not Solution.preserves_compact_accents(''.join(words), ''.join(target)):
                            continue
                    elif len(target) != len(hypotheses) or not all((Solution.compatible(h, w) for h, w in zip(hypotheses, target))):
                        continue
                    score = 60.0 if span.has_marker else 45.0
                    if mode == 'compact' and ''.join(words) == ''.join(target):
                        score = 95.0 if span.has_marker else 85.0
                    verified.append(Solution.Candidate(entry, span.start, span.end, mode, cost=1, score=score, context_marker=span.has_marker, reasons=(mode + '_verified',) + (('extension_recovery',) if span.is_extension_of else ())))
        return (retrieved, verified, {'counts': counts, 'limits': limits, 'exhausted': any(flags.values()), 'exhaustion_flags': flags}, fields, [c for c in suspicious if c.field in fields])

    @staticmethod
    def contradicts_tone(candidate: 'Candidate', tokens: 'list[Token]') -> bool:
        """Prevent ordinary edit deletion from overriding an explicit residual tone."""
        start = bisect_left(tokens, candidate.start, key=lambda token: token.start)
        end = bisect_left(tokens, candidate.end, lo=start, key=lambda token: token.start)
        window = tokens[start:end]
        words = [token.nfc for token in window if token.end <= candidate.end]
        targets = (candidate.entry.nfc_tokens, candidate.entry.body_nfc_tokens)
        if candidate.body_start is not None:
            for index, token in enumerate(window):
                if token.start <= candidate.body_start < token.end:
                    words = words[index:]
                    words[0] = Solution.normalize_nfc(token.text[candidate.body_start - token.start:]).casefold()
                    break
            targets = (candidate.entry.body_nfc_tokens,)
        for target in targets:
            if len(target) > len(words):
                continue
            observed = words[-len(target):]
            for mode in ('telex', 'vni'):
                for word, canonical in zip(observed, target):
                    h = Solution.decode_token(word, mode)
                    if h.tone and Solution.fold_accents(h.text) == Solution.fold_accents(canonical):
                        tones = {c for c in unicodedata.normalize('NFD', canonical) if c in Solution.TONES.values()}
                        if tones != {h.tone}:
                            return True
        return False

    # matcher.py
    class AddressMatcher:
        """Synchronous, standard-library address matcher implementing general and typing recovery."""

        def __init__(self, streets: Sequence[str] | Set[str]=(), wards: Sequence[str] | Set[str]=(), provinces: Sequence[str] | Set[str]=(), *, config: 'MatcherConfig | None'=None) -> None:
            if config is not None and (not isinstance(config, Solution.MatcherConfig)):
                raise TypeError(f'config must be a MatcherConfig, got {type(config).__name__}')
            self._catalog = Solution.Catalog(streets=streets, wards=wards, provinces=provinces)
            self._config = config if config is not None else Solution.MatcherConfig()
            self._approx_index = Solution.ApproximateIndex(self._catalog) if self._config.enable_fuzzy else None
            self._typing_index = Solution.TypingIndex(self._catalog) if self._config.enable_typing or self._config.enable_compact else None

        @property
        def catalog(self) -> 'Catalog':
            """Return the immutable catalog instance."""
            return self._catalog

        @property
        def config(self) -> 'MatcherConfig':
            """Return the matcher configuration."""
            return self._config

        def _is_full_exact_fast_path(self, address: str, tokens: 'list[Token]', exact_candidates: 'list[Candidate]') -> bool:
            """Check if address is completely explained by exact matches with no suspicious tails."""
            if not exact_candidates:
                return False
            intervals = sorted(((c.start, c.end) for c in exact_candidates))
            position, covered_end = (0, -1)
            for token in tokens:
                while position < len(intervals) and intervals[position][0] <= token.start:
                    covered_end = max(covered_end, intervals[position][1])
                    position += 1
                if covered_end < token.end and token.folded != 'so' and (not token.folded.replace('/', '').isdigit()):
                    return False
            return True

        def _execute_pipeline(self, address: str) -> 'tuple[dict[str, str | None], list[Candidate], list[Candidate], list[Candidate], dict[str, Any], dict[str, Any]]':
            """Internal unified execution pipeline for identify and identify_with_trace."""
            tokens = Solution.tokenize(address)
            if not tokens or self._catalog.max_tokens == 0:
                empty_result: dict[str, str | None] = {'street': None, 'ward': None, 'province': None}
                empty_selection: dict[str, Any] = {'best_score': 0.0, 'ambiguous_fields': [], 'pruned_fields': [], 'rejection_reasons': {'street': 'no_candidates', 'ward': 'no_candidates', 'province': 'no_candidates'}}
                empty_budgets: dict[str, Any] = {'counts': {'fuzzy_spans': 0, 'postings': 0, 'verifications': 0, 'candidates_per_span': 0}, 'limits': {'fuzzy_spans': self._config.max_fuzzy_spans, 'postings': self._config.max_postings, 'verifications': self._config.max_verifications, 'candidates_per_span': self._config.max_candidates_per_span}, 'exhausted': False}
                return (empty_result, [], [], [], empty_selection, empty_budgets)
            exact_retrieved = Solution.retrieve_candidates(tokens, self._catalog, self._config, address)
            if not (self._config.enable_fuzzy or self._typing_index) or self._is_full_exact_fast_path(address, tokens, exact_retrieved):
                all_retrieved = exact_retrieved
                all_verified = exact_retrieved
                budget_trace: dict[str, Any] = {'counts': {'fuzzy_spans': 0, 'postings': 0, 'verifications': 0, 'candidates_per_span': 0}, 'limits': {'fuzzy_spans': self._config.max_fuzzy_spans, 'postings': self._config.max_postings, 'verifications': self._config.max_verifications, 'candidates_per_span': self._config.max_candidates_per_span}, 'exhausted': False}
                result, retained, selection_trace = Solution.select_candidates(all_verified, self._config)
                return (result, all_retrieved, all_verified, retained, selection_trace, budget_trace)
            span_data = Solution.extract_spans(address, tokens, exact_retrieved, self._config.max_fuzzy_spans)
            fuzzy_retrieved, fuzzy_verified = ([], [])
            exhausted_fields, suspicious_to_abstain = (set(), [])
            budget_trace = {'counts': {'fuzzy_spans': 0, 'postings': 0, 'verifications': 0, 'candidates_per_span': 0}, 'limits': {'fuzzy_spans': self._config.max_fuzzy_spans, 'postings': self._config.max_postings, 'verifications': self._config.max_verifications, 'candidates_per_span': self._config.max_candidates_per_span}, 'exhausted': False, 'exhaustion_flags': {}}
            if self._config.enable_fuzzy:
                fuzzy_config = self._config
                if self._typing_index is not None:
                    fuzzy_config = replace(self._config, max_candidates_per_span=max(1, self._config.max_candidates_per_span // 2))
                fuzzy_retrieved, fuzzy_verified, budget_trace, exhausted_fields, suspicious_to_abstain = Solution.recover_fuzzy_candidates(span_data, fuzzy_config, self._approx_index)
                budget_trace['limits']['candidates_per_span'] = self._config.max_candidates_per_span
            if self._typing_index is not None:
                remaining_postings = self._config.max_postings - budget_trace['counts']['postings']
                remaining_checks = self._config.max_verifications - budget_trace['counts']['verifications']
                remaining_candidates = self._config.max_candidates_per_span - budget_trace['counts']['candidates_per_span']
                if remaining_postings > 0 and remaining_checks > 0 and (remaining_candidates > 0):
                    typing_config = replace(self._config, max_postings=remaining_postings, max_verifications=remaining_checks, max_candidates_per_span=remaining_candidates)
                    retrieved, verified, typing_budget, fields, suspicious = Solution.recover_typing(address, span_data, self._typing_index, typing_config)
                    fuzzy_retrieved.extend(retrieved)
                    fuzzy_verified.extend(verified)
                    exhausted_fields.update(fields)
                    suspicious_to_abstain.extend(suspicious)
                    for key, count in typing_budget['counts'].items():
                        if key in ('postings', 'verifications', 'candidates_per_span'):
                            budget_trace['counts'][key] += count
                        else:
                            budget_trace['counts'][key] = max(budget_trace['counts'][key], count)
                    budget_trace['exhausted'] |= typing_budget['exhausted']
                    for key, value in typing_budget['exhaustion_flags'].items():
                        budget_trace['exhaustion_flags'][key] = budget_trace['exhaustion_flags'].get(key, False) or value
                else:
                    _, _, suspicious = span_data
                    suspicious_to_abstain.extend(suspicious)
                    exhausted_fields.update((c.field for c in fuzzy_verified))
                    exhausted_fields.update((c.field for c in suspicious))
                    budget_trace['exhausted'] = True
                    for name, remaining in (('postings', remaining_postings), ('verifications', remaining_checks), ('candidates_per_span', remaining_candidates)):
                        if remaining == 0:
                            budget_trace['exhaustion_flags'][name] = True
            if self._config.enable_typing:
                fuzzy_verified = [c for c in fuzzy_verified if c.path != 'fuzzy' or not Solution.contradicts_tone(c, tokens)]
            all_retrieved = exact_retrieved + fuzzy_retrieved
            abstain_exact_ids = {id(c) for c in suspicious_to_abstain}
            survived_exact = [c for c in exact_retrieved if id(c) not in abstain_exact_ids]
            survived_fuzzy = [c for c in fuzzy_verified if c.field not in exhausted_fields]
            deduped = {}
            for candidate in survived_exact + survived_fuzzy:
                key = (candidate.entry.entry_id, candidate.start, candidate.end)
                if key not in deduped or candidate.score > deduped[key].score:
                    deduped[key] = candidate
            all_verified = list(deduped.values())
            result, retained, selection_trace = Solution.select_candidates(all_verified, self._config)
            for f in exhausted_fields:
                if result[f] is None:
                    selection_trace['rejection_reasons'][f] = 'budget_exhausted'
            return (result, all_retrieved, all_verified, retained, selection_trace, budget_trace)

        def identify(self, address: str) -> dict[str, str | None]:
            """Identify canonical street, ward, and province from free-form address string.

            Returns a fresh dictionary with exact keys 'street', 'ward', 'province'.
            Does not serialize detailed candidate traces. Parity with identify_with_trace.
            """
            if not isinstance(address, str):
                raise TypeError(f'address must be a str, got {type(address).__name__}')
            result, _, _, _, _, _ = self._execute_pipeline(address)
            return {'street': result['street'], 'ward': result['ward'], 'province': result['province']}


In [ ]:
# Download public test
!rm -rf test.json
!gdown --fuzzy https://drive.google.com/file/d/18qzaHmx9-fcKPi4gTq304G2JvXy_g6XW/view?usp=sharing -O test.json
!gdown --fuzzy https://drive.google.com/file/d/1PHRkRLRaO2vFivLbcPJsaxjvZ7K_I3lU/view?usp=drive_link -O list_province.txt
!gdown --fuzzy https://drive.google.com/file/d/1QrSouYn6IUEW7Impje1vOxzsksjlAJ-h/view?usp=drive_link -O list_ward.txt
!gdown --fuzzy https://drive.google.com/file/d/1yP2dBE8K9QtUiaLPewtQRr-hDH0gOgfa/view?usp=drive_link -O list_street.txt

In [ ]:
def _judge_normalize(s: str, context=None):
    return s

EXCEL_FILE = f'{TEAM_NAME}.xlsx'

import json
import time
with open('test.json') as f:
    data = json.load(f)

with open("list_province.txt") as f:
    list_province = f.read().strip().splitlines()

with open("list_ward.txt") as f:
    list_ward = f.read().strip().splitlines()

with open("list_street.txt") as f:
    list_street = f.read().strip().splitlines()

summary_only = True
df = []
solution = Solution(
    list_province=list_province,
    list_ward=list_ward,
    list_street=list_street,
)
timer = []
correct = 0
for test_idx, data_point in enumerate(data):
    address = data_point["address"]

    ok = 0
    try:
        answer = data_point["result"]
        answer["province_normalized"] = _judge_normalize(answer["province"])
        answer["ward_normalized"] = _judge_normalize(answer["ward"])
        answer["street_normalized"] = _judge_normalize(answer["street"])

        start = time.perf_counter_ns()
        result = solution.process(address)
        finish = time.perf_counter_ns()
        timer.append(finish - start)
        result["province_normalized"] = _judge_normalize(result["province"])
        result["ward_normalized"] = _judge_normalize(result["ward"])
        result["street_normalized"] = _judge_normalize(result["street"])

        province_correct = int(answer["province_normalized"] == result["province_normalized"])
        ward_correct = int(answer["ward_normalized"] == result["ward_normalized"])
        street_correct = int(answer["street_normalized"] == result["street_normalized"])
        ok = province_correct + street_correct + ward_correct

        df.append([
            test_idx,
            address,

            answer["province"],
            result["province"],
            answer["province_normalized"],
            result["province_normalized"],
            province_correct,

            answer["ward"],
            result["ward"],
            answer["ward_normalized"],
            result["ward_normalized"],
            ward_correct,

            answer["street"],
            result["street"],
            answer["street_normalized"],
            result["street_normalized"],
            street_correct,

            ok,
            timer[-1] / 1_000_000_000,
        ])
    except Exception as e:
        print(f"{answer = }")
        print(f"{result = }")
        df.append([
            test_idx,
            address,

            answer["province"],
            "EXCEPTION",
            answer["province_normalized"],
            "EXCEPTION",
            0,

            answer["ward"],
            "EXCEPTION",
            answer["ward_normalized"],
            "EXCEPTION",
            0,

            answer["street"],
            "EXCEPTION",
            answer["street_normalized"],
            "EXCEPTION",
            0,

            0,
            0,
        ])
        # any failure count as a zero correct
        pass
    correct += ok


    if not summary_only:
        # responsive stuff
        print(f"Test {test_idx:5d}/{len(data):5d}")
        print(f"Correct: {ok}/3")
        print(f"Time Executed: {timer[-1] / 1_000_000_000:.4f}")


print(f"-"*30)
total = len(data) * 3
score_scale_10 = round(correct / total * 10, 2)
if len(timer) == 0:
    timer = [0]
max_time_sec = round(max(timer) / 1_000_000_000, 4)
avg_time_sec = round((sum(timer) / len(timer)) / 1_000_000_000, 4)

import pandas as pd

df2 = pd.DataFrame(
    [[correct, total, score_scale_10, max_time_sec, avg_time_sec]],
    columns=['correct', 'total', 'score / 10', 'max_time_sec', 'avg_time_sec',],
)

columns = [
    'ID',
    'text',

    'province',
    'province_student',
    'province_normalized',
    'province_student_normalized',
    'province_correct',

    'ward',
    'ward_student',
    'ward_normalized',
    'ward_student_normalized',
    'ward_correct',

    'street',
    'street_student',
    'street_normalized',
    'street_student_normalized',
    'street_correct',

    'total_correct',
    'time_sec',
]

df = pd.DataFrame(df)
df.columns = columns

print(f'{TEAM_NAME = }')
print(f'{EXCEL_FILE = }')
print(df2)

!pip install xlsxwriter
writer = pd.ExcelWriter(EXCEL_FILE, engine='xlsxwriter')
df2.to_excel(writer, index=False, sheet_name='summary')
df.to_excel(writer, index=False, sheet_name='details')
writer.close()